# Reusable Regression Workflow — Notebook Template

This notebook is a compact **EDA and validation reference**. It uses generated **synthetic** data and calls the reusable package rather than reproducing a long one-off notebook.

**Run from the repository root**, after `pip install -e '.[notebooks]'` and `python scripts/make_demo_data.py`. In JupyterLab open this notebook; paths are resolved from either the project root or `notebooks/`.


## 1. Load, inspect and reserve holdout

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from ds_starter.config import load_config
from ds_starter.data import load_table, development_holdout_indices, build_target
from ds_starter.eda import data_quality_report, top_correlations
from ds_starter.cleaning import apply_fixed_rules
from ds_starter.features import create_features, model_inputs
from ds_starter.validation import cross_validate_models

root = Path.cwd()
if not (root / "configs").exists():
    root = root.parent
config = load_config(root / "configs/tabular_demo.yml")
data_file = root / config["data"]["train_path"]
raw = load_table(data_file)
train_idx, holdout_idx = development_holdout_indices(raw, config)
development = raw.iloc[train_idx].copy()
print("development:", development.shape, "held-out rows:", len(holdout_idx))

## 2. EDA — development data only

In [ ]:
display(data_quality_report(development).sort_values("missing_pct", ascending=False))
display(top_correlations(development, exclude=[config["target_column"]]).head(10))
development[config["target_column"]].plot.hist(bins=25, title="Target distribution")
plt.show()

## 3. Clean and engineer features

These are fixed rules and deterministic features; *group-imputation means, encoding, and scaling are not fitted here*. They will be learned inside each training fold.

In [ ]:
clean = apply_fixed_rules(raw, config.get("cleaning"))
engineered = create_features(clean, config)
X = model_inputs(engineered, config)
y = build_target(engineered, config)
X_dev, y_dev = X.iloc[train_idx], y.iloc[train_idx]
display(X_dev.head())

## 4. Group-aware cross-validation

In [ ]:
groups = engineered.iloc[train_idx][config["data"]["group_column"]]
fold_metrics, summary = cross_validate_models(X_dev, y_dev, config, groups=groups)
display(fold_metrics.round(2))
display(summary.round(2))

## 5. Final holdout scoring and saved artifacts

The high-level runner repeats the full experiment, refits the validation winner, and evaluates the holdout only **once**. Do not use the holdout to choose different hyperparameters.

In [ ]:
import os
from ds_starter.runner import run_experiment

previous_dir = Path.cwd()
try:
    os.chdir(root)  # config paths are relative to project root
    result = run_experiment("configs/tabular_demo.yml")
finally:
    os.chdir(previous_dir)
print("chosen:", result["chosen_model"])
display(result["holdout_scores"].round(2))

## Next: adapt to your own problem

- Change the YAML feature lists, cleaning rules, target, holdout and CV strategy.
- For chronological observations, use `configs/timeseries_demo.yml`.
- For one-off domain logic, create a small adapter function, keep it in `src/`, and test it.
- See `docs/resource_catalog.md` for official code examples and Google search queries.